# Conversores Boost (Elevador)
Representación del comportamiento de un convertidor conmutado, mostrando las variaciones de corriente y voltaje en sus componentes principales: el switch, el diodo, la bobina y el capacitor.

El objetivo principal del Boost es analizar la respuesta del sistema en función de parámetros como la inductancia (L), la capacitancia (C) y la resistencia de carga (R). A través de la gráfica interactiva, se pueden observar los efectos de la conmutación en la transferencia de energía, visualizando el comportamiento del sistema en el dominio del tiempo.

### Explicación de la gráfica
La gráfica muestra tres aspectos clave del convertidor:

- Corrientes en el sistema 
- Corriente que pasa por el switch. (𝑖𝑆𝑤)
- Corriente que pasa por el diodo.(𝑖𝐷)
- Corriente a través de la bobina.  (𝑖𝐿)

➝ Nos ayuda a visualizar el efecto del ciclo de trabajo (𝐷) y observar cómo varían las corrientes con la conmutación.

### Respuesta del filtro RC

Muestra la salida del convertidor tras pasar por el filtro LC-RC, simulando cómo se suavizan las oscilaciones del voltaje para obtener una señal continua.
Voltajes en los componentes

- Voltaje en el switch. (𝑣𝑆𝑤)
- Voltaje en el diodo.(𝑣𝐷)
​- Voltaje en la bobina (𝑣𝐿)

➝ Nos permite entender cómo se distribuye la tensión en los diferentes elementos del circuito durante el proceso de conmutación.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import bilinear, lfilter
import ipywidgets as widgets
from IPython.display import display

# Parámetros iniciales
D = 2 / 3
vi = 10
Fsw = 1e4  # 10 kHz
Tsw = 1 / Fsw

# Contenedor para la salida de gráficos
output = widgets.Output()

# --- Función para calcular corrientes y voltajes ---
def calcular_senales(L, C, R, ciclos):
    vo = vi / (1 - D)
    ioDC = vo / R
    iLDC = ioDC / (1 - D)
    delta_iL = vi * D * Tsw / L

    imax = iLDC + delta_iL / 2
    imin = iLDC - delta_iL / 2

    # Señales de corriente
    puntos = 10000
    puntos_D = int(np.floor(D * puntos))
    puntos_1_D = puntos - puntos_D

    iSw = np.zeros(puntos)
    iSw[:puntos_D] = np.linspace(imin, imax, puntos_D)
    iSw = np.tile(iSw, ciclos)

    iD = np.zeros(puntos)
    iD[puntos_D:] = np.linspace(imax, imin, puntos_1_D)
    iD = np.tile(iD, ciclos)

    iL = iSw + iD  

    # Eje temporal
    t = np.linspace(0, len(iSw) * Tsw / puntos, len(iSw))

    # Respuesta del filtro RC
    num, den = bilinear([R], [R * C, 1], Fsw * puntos)
    vo = lfilter(num, den, iD)

    return t, iSw, iD, iL, vo

# --- Función para análisis de frecuencia ---
def analizar_frecuencia(signal, fs):
    N = len(signal)
    f = np.fft.rfftfreq(N, d=1/fs)
    fft_signal = np.abs(np.fft.rfft(signal)) / N
    return f, fft_signal

# --- Función para graficar ---
def graficar(t, iSw, iD, iL, vo, opcion_transitoria):
    with output:
        output.clear_output(wait=True)
        plt.figure(figsize=(12, 8))

        # Gráfica de corrientes
        plt.subplot(3, 1, 1)
        plt.plot(t, iSw, label="Corriente por el switch (iSw)", color='b')
        plt.plot(t, iD, label="Corriente por el diodo (iD)", color='r')
        plt.plot(t, iL, label="Corriente por la bobina (iL)", color='g')
        plt.xlim([0, 4 * Tsw])
        plt.title("Corrientes del sistema")
        plt.xlabel("Tiempo [s]")
        plt.ylabel("Corriente [A]")
        plt.legend()
        plt.grid()

        # Gráfica de respuesta del filtro RC
        plt.subplot(3, 1, 2)

        if opcion_transitoria == "Transitoria":
            plt.plot(t[:len(t)//5], vo[:len(vo)//5], label="Respuesta transitoria", color='m')
            plt.xlim([0, 4 * Tsw])
        else:
            plt.plot(t[-len(t)//5:], vo[-len(vo)//5:], label="Respuesta no transitoria", color='c')
            plt.xlim([t[-len(t)//5], t[-1]])

        plt.title(f"Respuesta del filtro RC ({opcion_transitoria})")
        plt.xlabel("Tiempo [s]")
        plt.ylabel("Voltaje [V]")
        plt.legend()
        plt.grid()

        # Análisis en frecuencia
        plt.subplot(3, 1, 3)
        fs = 1 / (t[1] - t[0])
        f_iD, fft_iD = analizar_frecuencia(iD, fs)
        f_vo, fft_vo = analizar_frecuencia(vo, fs)

        plt.plot(f_iD, fft_iD, label="Espectro de la señal cuadrada (iD)", color='orange')
        plt.plot(f_vo, fft_vo, label="Espectro del voltaje filtrado (vo)", color='purple')
        plt.title("Análisis en frecuencia")
        plt.xlabel("Frecuencia [Hz]")
        plt.ylabel("Magnitud")
        plt.legend()
        plt.grid()
        plt.tight_layout()
        plt.xlim([0, Fsw * 10])
        plt.show()

# --- Widgets interactivos ---
titulo = widgets.HTML(value="<h2 style='text-align:left; color: Teal;'>Simulador Conversores Boost (Elevador)</h2>")
descripcion = widgets.HTML(value="<p style='text-align:justify;'>Visualiza el comportamiento de un conversor Boost (Elevador) y analiza sus principales variables eléctricas. A través de las gráficas, podrás observar la evolución de las corrientes en el interruptor, el diodo y la bobina, así como la respuesta del voltaje de salida. Además, el análisis en frecuencia te permitirá comparar el espectro de la señal conmutada y la respuesta filtrada, ayudando a comprender la eficiencia y el desempeño del conversor</p>")

L_input = widgets.FloatText(value=3e-3, description="Inductancia:", step=1e-4, layout=widgets.Layout(width="200px"))
C_input = widgets.FloatText(value=20e-6, description="Capacitancia:", step=1e-6, layout=widgets.Layout(width="200px"))
R_input = widgets.FloatText(value=20, description="Resistencia:", step=1, layout=widgets.Layout(width="200px"))
frecuencia_input = widgets.FloatText(value=10e3, description="Frecuencia:", step=1e3, layout=widgets.Layout(width="200px"))

transitoria_selector = widgets.ToggleButtons(
    options=["Transitoria", "Estacionaria"],
    description="Opciones:",
    button_style="info"
)

calcular_btn = widgets.Button(description="Iniciar Simulación ▶️", button_style="success")

# --- Función de actualización ---
def actualizar(change=None):
    global Fsw, Tsw
    L = L_input.value
    C = C_input.value
    R = R_input.value
    Fsw = frecuencia_input.value
    Tsw = 1 / Fsw

    t, iSw, iD, iL, vo = calcular_senales(L, C, R, ciclos=22)  
    graficar(t, iSw, iD, iL, vo, transitoria_selector.value)

# Conexión del botón al cálculo
calcular_btn.on_click(actualizar)

# --- Diseño de la interfaz ---
entrada_datos = widgets.HBox([L_input, C_input, R_input, frecuencia_input])
controles = widgets.VBox([titulo, descripcion, entrada_datos, transitoria_selector, calcular_btn, output])

# Mostrar la interfaz
display(controles)